<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/Criar_agente_de_e_mails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U langgraph
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U pydantic
!pip install -q -U python-dotenv

In [3]:
import os
import json
from typing import TypedDict, Literal, Optional, List, Dict, Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent

In [4]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("GEMINI_API_KEY carregada pelo Colab Secrets.")
except Exception:
    load_dotenv()
    print("Tentando carregar variáveis pelo arquivo .env.")

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY não encontrada.")

GEMINI_API_KEY carregada pelo Colab Secrets.


In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [6]:
#Definir perfil do usuário e regra de triagem
perfil_usuario = {
    "nome": "Leandro Moquiuti",
    "cargo": "Desenvolvedor Backend Java Sênior",
    "empresa": "Marlabs",
    "prioridades": [
        "assuntos técnicos de projetos",
        "reuniões com cliente",
        "demandas de liderança",
        "arquitetura de software",
        "atividades urgentes do time"
    ],
    "tom_resposta": "profissional, objetivo, educado e direto",
    "horario_preferencial_reunioes": "entre 09:00 e 17:00"
}

In [7]:
regras_triagem = {
    "responder": [
        "quando o e-mail exigir retorno direto",
        "quando houver pergunta técnica",
        "quando for solicitação de cliente, liderança ou time",
        "quando pedir posicionamento, validação ou confirmação"
    ],
    "notificar": [
        "quando o e-mail for importante, mas não exigir resposta imediata",
        "quando for apenas informativo, porém relevante",
        "quando mencionar prazo, reunião, cliente ou bloqueio"
    ],
    "ignorar": [
        "quando for propaganda",
        "quando for newsletter genérica",
        "quando não tiver relação com trabalho ou prioridade",
        "quando não exigir nenhuma ação"
    ]
}

In [8]:
#simular entrada de email como envento gatilho

email_evento = {
    "remetente": "ana.cliente@empresa.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Dúvida sobre integração B2B com fornecedor",
    "corpo": """
Olá, Leandro.

Conseguimos avançar na análise da integração B2B, mas surgiu uma dúvida sobre o fluxo de autenticação
e sobre qual endpoint será usado para consultar disponibilidade.

Você consegue nos responder ainda hoje ou sugerir um horário para alinharmos rapidamente?

Obrigado.
""",
    "data_recebimento": "2026-05-18 14:30"
}

In [9]:
#modelo pydantic para racioncínio e classificação

class ClassificacaoEmail(BaseModel):
    raciocinio: str = Field(
        description="Explicação breve do motivo da classificação."
    )
    classificacao: Literal["responder", "notificar", "ignorar"] = Field(
        description="Decisão de triagem para o e-mail."
    )
    prioridade: Literal["baixa", "media", "alta"] = Field(
        description="Prioridade percebida do e-mail."
    )
    motivo: str = Field(
        description="Motivo objetivo da decisão."
    )

In [10]:
#promt do sistema

def montar_prompt_triagem(perfil: dict, regras: dict) -> str:
    return f"""
Você é um agente de triagem de e-mails.

Seu trabalho é classificar e-mails recebidos em uma das três categorias:

1. responder
2. notificar
3. ignorar

Perfil do usuário:
{json.dumps(perfil, ensure_ascii=False, indent=2)}

Regras de triagem:
{json.dumps(regras, ensure_ascii=False, indent=2)}

Critérios:
- Classifique como "responder" se o e-mail exigir resposta direta.
- Classifique como "notificar" se for importante, mas não exigir resposta imediata.
- Classifique como "ignorar" se for irrelevante, promocional ou sem ação necessária.
- Seja objetivo.
- Não invente informações.
- Use o perfil do usuário para decidir relevância e prioridade.
"""

In [11]:
prompt_triagem = montar_prompt_triagem(perfil_usuario, regras_triagem)

print(prompt_triagem)


Você é um agente de triagem de e-mails.

Seu trabalho é classificar e-mails recebidos em uma das três categorias:

1. responder
2. notificar
3. ignorar

Perfil do usuário:
{
  "nome": "Leandro Moquiuti",
  "cargo": "Desenvolvedor Backend Java Sênior",
  "empresa": "Marlabs",
  "prioridades": [
    "assuntos técnicos de projetos",
    "reuniões com cliente",
    "demandas de liderança",
    "arquitetura de software",
    "atividades urgentes do time"
  ],
  "tom_resposta": "profissional, objetivo, educado e direto",
  "horario_preferencial_reunioes": "entre 09:00 e 17:00"
}

Regras de triagem:
{
  "responder": [
    "quando o e-mail exigir retorno direto",
    "quando houver pergunta técnica",
    "quando for solicitação de cliente, liderança ou time",
    "quando pedir posicionamento, validação ou confirmação"
  ],
  "notificar": [
    "quando o e-mail for importante, mas não exigir resposta imediata",
    "quando for apenas informativo, porém relevante",
    "quando mencionar prazo, 

In [12]:
#LLM estruturado para triagem
llm_triagem = llm.with_structured_output(ClassificacaoEmail)

In [13]:
acoes_executadas = []

In [14]:
@tool
def WriteMail(destinatario: str, assunto: str, corpo: str) -> str:
    """
    Cria um rascunho de e-mail para o destinatário informado.
    Não envia o e-mail de verdade.
    """
    acao = {
        "tipo": "rascunho_email",
        "destinatario": destinatario,
        "assunto": assunto,
        "corpo": corpo
    }

    acoes_executadas.append(acao)

    return f"Rascunho criado para {destinatario} com assunto '{assunto}'."

In [15]:
@tool
def ScheduleMeeting(participante: str, data_hora: str, assunto: str) -> str:
    """
    Agenda uma reunião simulada com o participante informado.
    Não cria evento real em calendário.
    """
    acao = {
        "tipo": "reuniao",
        "participante": participante,
        "data_hora": data_hora,
        "assunto": assunto
    }

    acoes_executadas.append(acao)

    return f"Reunião simulada criada com {participante} em {data_hora} sobre '{assunto}'."

In [16]:
@tool
def CheckCalendar(data: str) -> str:
    """
    Consulta disponibilidade simulada do calendário para uma data.
    """
    disponibilidades = {
        "hoje": "Disponível às 16:00 e 16:30.",
        "amanha": "Disponível às 10:00, 14:00 e 15:30.",
        "2026-05-18": "Disponível às 16:00 e 16:30."
    }

    return disponibilidades.get(
        data.lower(),
        "Disponibilidade simulada: 10:00, 14:00 ou 16:00."
    )

In [17]:
tools = [
    WriteMail,
    ScheduleMeeting,
    CheckCalendar
]

In [18]:
#Prompt do agente ReAct de e-mail

system_prompt_agent = f"""
Você é um agente assistente de e-mails para {perfil_usuario["nome"]}.

Perfil:
{json.dumps(perfil_usuario, ensure_ascii=False, indent=2)}

Você possui ferramentas disponíveis:
- WriteMail: criar rascunho de resposta.
- ScheduleMeeting: simular agendamento de reunião.
- CheckCalendar: verificar disponibilidade simulada.

Regras:
- Não envie e-mail real.
- Sempre crie apenas rascunhos.
- Se o e-mail pedir resposta, use WriteMail.
- Se o e-mail sugerir reunião ou alinhamento, consulte CheckCalendar antes.
- Se houver necessidade de reunião, use ScheduleMeeting após consultar disponibilidade.
- Seja profissional, objetivo e educado.
- O tom deve parecer com o perfil do usuário.
"""

In [19]:
email_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt_agent
)

/tmp/ipykernel_1511/367446719.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  email_agent = create_react_agent(


In [20]:
class EmailState(TypedDict):
    email: Dict[str, Any]
    classificacao: Optional[str]
    prioridade: Optional[str]
    raciocinio: Optional[str]
    motivo: Optional[str]
    resposta_agente: Optional[str]
    notificacao: Optional[str]

In [21]:
def no_triagem(state: EmailState):
    email = state["email"]

    prompt_usuario = f"""
Classifique o seguinte e-mail:

Remetente: {email["remetente"]}
Destinatário: {email["destinatario"]}
Assunto: {email["assunto"]}
Data: {email["data_recebimento"]}

Corpo:
{email["corpo"]}
"""

    resultado = llm_triagem.invoke([
        SystemMessage(content=prompt_triagem),
        HumanMessage(content=prompt_usuario)
    ])

    return {
        "classificacao": resultado.classificacao,
        "prioridade": resultado.prioridade,
        "raciocinio": resultado.raciocinio,
        "motivo": resultado.motivo
    }

In [22]:
def triagem_router(state: EmailState):
    classificacao = state.get("classificacao")

    if classificacao == "responder":
        return "responder"

    if classificacao == "notificar":
        return "notificar"

    return "ignorar"

In [23]:
def no_responder(state: EmailState):
    email = state["email"]

    mensagem = f"""
Você recebeu este e-mail e precisa tratar a demanda.

Classificação: {state["classificacao"]}
Prioridade: {state["prioridade"]}
Motivo: {state["motivo"]}

E-mail:
Remetente: {email["remetente"]}
Assunto: {email["assunto"]}
Corpo:
{email["corpo"]}

Tarefa:
Crie um rascunho de resposta profissional.
Se o e-mail pedir alinhamento ou reunião, consulte o calendário antes e sugira/agende um horário simulado.
"""

    resultado = email_agent.invoke({
        "messages": [
            HumanMessage(content=mensagem)
        ]
    })

    resposta_final = resultado["messages"][-1].content

    return {
        "resposta_agente": resposta_final
    }

In [24]:
def no_notificar(state: EmailState):
    email = state["email"]

    notificacao = f"""
Notificação importante:

Assunto: {email["assunto"]}
Remetente: {email["remetente"]}
Prioridade: {state["prioridade"]}
Motivo: {state["motivo"]}

Resumo:
Este e-mail foi considerado relevante, mas não exige resposta imediata.
"""

    return {
        "notificacao": notificacao.strip()
    }

In [25]:
def no_ignorar(state: EmailState):
    return {
        "notificacao": "E-mail ignorado conforme regras de triagem."
    }

In [26]:
#montar grafo
builder = StateGraph(EmailState)

builder.add_node("triagem", no_triagem)
builder.add_node("responder", no_responder)
builder.add_node("notificar", no_notificar)
builder.add_node("ignorar", no_ignorar)

builder.set_entry_point("triagem")

builder.add_conditional_edges(
    "triagem",
    triagem_router,
    {
        "responder": "responder",
        "notificar": "notificar",
        "ignorar": "ignorar"
    }
)

builder.add_edge("responder", END)
builder.add_edge("notificar", END)
builder.add_edge("ignorar", END)

grafo_email = builder.compile()

In [27]:
#testando fluxo completo
estado_inicial = {
    "email": email_evento,
    "classificacao": None,
    "prioridade": None,
    "raciocinio": None,
    "motivo": None,
    "resposta_agente": None,
    "notificacao": None
}

resultado = grafo_email.invoke(estado_inicial)

resultado

{'email': {'remetente': 'ana.cliente@empresa.com',
  'destinatario': 'leandro@empresa.com',
  'assunto': 'Dúvida sobre integração B2B com fornecedor',
  'corpo': '\nOlá, Leandro.\n\nConseguimos avançar na análise da integração B2B, mas surgiu uma dúvida sobre o fluxo de autenticação\ne sobre qual endpoint será usado para consultar disponibilidade.\n\nVocê consegue nos responder ainda hoje ou sugerir um horário para alinharmos rapidamente?\n\nObrigado.\n',
  'data_recebimento': '2026-05-18 14:30'},
 'classificacao': 'responder',
 'prioridade': 'alta',
 'raciocinio': "O e-mail é de um cliente, contém perguntas técnicas sobre um projeto e solicita uma resposta direta ou agendamento de reunião, o que se alinha às prioridades do usuário e às regras de triagem para 'responder'.",
 'motivo': 'Solicitação de cliente com perguntas técnicas e pedido de retorno direto/agendamento.',
 'resposta_agente': [{'type': 'text',
   'text': 'Rascunho de e-mail criado:\n\n**Para:** ana.cliente@empresa.com\n

In [28]:
#fiz assim pra ter uma visualização melhor

print("Classificação:", resultado["classificacao"])
print("Prioridade:", resultado["prioridade"])
print("Raciocínio:", resultado["raciocinio"])
print("Motivo:", resultado["motivo"])

print("\nResposta do agente:")
print(resultado["resposta_agente"])

print("\nNotificação:")
print(resultado["notificacao"])

print("\nAções executadas:")
print(json.dumps(acoes_executadas, ensure_ascii=False, indent=2))

Classificação: responder
Prioridade: alta
Raciocínio: O e-mail é de um cliente, contém perguntas técnicas sobre um projeto e solicita uma resposta direta ou agendamento de reunião, o que se alinha às prioridades do usuário e às regras de triagem para 'responder'.
Motivo: Solicitação de cliente com perguntas técnicas e pedido de retorno direto/agendamento.

Resposta do agente:
[{'type': 'text', 'text': 'Rascunho de e-mail criado:\n\n**Para:** ana.cliente@empresa.com\n**Assunto:** Re: Dúvida sobre integração B2B com fornecedor\n**Corpo:**\n\nOlá, Ana.\n\nRecebi sua mensagem sobre as dúvidas na integração B2B. Para alinharmos rapidamente e endereçarmos os pontos sobre o fluxo de autenticação e o endpoint de consulta de disponibilidade, agendei uma reunião para hoje, 26/07, às 14:00.\n\nFico à disposição.\n\nAtenciosamente,\nLeandro Moquiuti', 'extras': {'signature': 'Cs0DAQw51sdpcdzfusPjOI+Kx1u7/wADCM082Lj2KIsza1StO6qqXfQqCagWlNQgwz81/FKAkNgdnz6qMC4LcI3kJPmoVOSpddpaQq88u4Io/cbMITcphUVlCWf

In [29]:
#esse teste deve ser ignorado

email_promocional = {
    "remetente": "marketing@lojaqualquer.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Promoção imperdível de celulares",
    "corpo": """
Olá! Temos ofertas incríveis de smartphones, acessórios e eletrônicos.
Clique agora para aproveitar.
""",
    "data_recebimento": "2026-05-18 15:00"
}

In [30]:
resultado_promocional = grafo_email.invoke({
    "email": email_promocional,
    "classificacao": None,
    "prioridade": None,
    "raciocinio": None,
    "motivo": None,
    "resposta_agente": None,
    "notificacao": None
})

print("Classificação:", resultado_promocional["classificacao"])
print("Prioridade:", resultado_promocional["prioridade"])
print("Motivo:", resultado_promocional["motivo"])
print("Notificação:", resultado_promocional["notificacao"])

Classificação: ignorar
Prioridade: baixa
Motivo: É um e-mail de propaganda/newsletter genérica e não exige nenhuma ação ou resposta relacionada ao trabalho.
Notificação: E-mail ignorado conforme regras de triagem.


In [31]:
#esse test é com email informativo

email_informativo = {
    "remetente": "comunicados@marlabs.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Comunicado sobre atualização de política interna",
    "corpo": """
Olá,

Informamos que a política interna de segurança será atualizada na próxima semana.
Não é necessária nenhuma ação imediata, mas recomendamos a leitura do material enviado.

Atenciosamente,
Equipe de Comunicação Interna
""",
    "data_recebimento": "2026-05-18 16:00"
}

In [32]:
resultado_informativo = grafo_email.invoke({
    "email": email_informativo,
    "classificacao": None,
    "prioridade": None,
    "raciocinio": None,
    "motivo": None,
    "resposta_agente": None,
    "notificacao": None
})

print("Classificação:", resultado_informativo["classificacao"])
print("Prioridade:", resultado_informativo["prioridade"])
print("Motivo:", resultado_informativo["motivo"])
print("Notificação:", resultado_informativo["notificacao"])

Classificação: notificar
Prioridade: media
Motivo: É um comunicado informativo e relevante da empresa que não demanda resposta direta, mas recomenda leitura.
Notificação: Notificação importante:

Assunto: Comunicado sobre atualização de política interna
Remetente: comunicados@marlabs.com
Prioridade: media
Motivo: É um comunicado informativo e relevante da empresa que não demanda resposta direta, mas recomenda leitura.

Resumo:
Este e-mail foi considerado relevante, mas não exige resposta imediata.
